# Week 10: Preprocessing Is Part of the Model

This notebook follows the reviewed Week 10 presentation. Read each concept and calculate the worked example before running its code.

## Lesson map

1. Preprocessing Is Part of the Model
2. Imputation Learns a Replacement Rule
3. Scaling Changes Units, Not Row Meaning
4. Encoding Represents Categories Numerically
5. ColumnTransformer Applies Rules by Column Type
6. Pipeline Gives One Fit and Predict Boundary
7. Pipelines Prevent Preprocessing Leakage
8. Cross-Validation Measures Variation Across Splits
9. Model Comparison Must Use the Same Evidence
10. Reproducibility Requires More Than a Random Seed
11. Persist the Whole Artifact and Load It Carefully
12. Guided Lab: Build a Leakage-Safe Pipeline

Use the same reasoning loop throughout: **predict, run, inspect, explain**.


## 1. Preprocessing Is Part of the Model

Raw tables often need **preprocessing** before a model can use them:

- impute missing values;
- scale numerical features;
- encode categories;
- select and order columns.

These steps learn parameters from training data, so they are part of the fitted model. Production inference must apply the same fitted transformations in the same order.

A **pipeline** binds preprocessing and prediction into one object with a shared `fit()` and `predict()` contract.

### Work it out first

Training learns:

- median age `34`;
- salary mean and standard deviation;
- category mapping for city.

New rows must use those stored training values. Recalculating them on each request changes the model.

### Notebook bridge

The model-development notebook builds and saves a complete preprocessing and model pipeline.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)

Expected output:

```text
Preprocessing is fitted on X_train and reused before every prediction.
```


## 2. Imputation Learns a Replacement Rule

**Imputation** replaces declared missing values using a rule.

Common simple rules:

- numerical median;
- numerical mean;
- most frequent category;
- constant marker such as `"missing"`.

The replacement value is a learned parameter. Fit it on training data, then apply the same value to validation, test, and production rows.

Imputation does not recover the unknown truth; it creates a usable value under an explicit assumption.

### Work it out first

Training ages `[20, 30, missing, 100]`

Observed sorted values `[20,30,100]`; median `30`.  
Imputed training ages `[20,30,30,100]`.

If test ages are `[missing, 40]`, use training median `30`, not the test median.

### Notebook bridge

Learners should inspect missing markers and justify each imputation strategy.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
imputer.fit(X_train[["age"]])

Expected output:

```text
The fitted imputer stores the training-column median.
```


## 3. Scaling Changes Units, Not Row Meaning

**Standardization** transforms a numerical value:

`z = (x - μ) / σ`

- `x`: original value
- `μ`: training-column mean
- `σ`: training-column standard deviation
- `z`: standardized value

After fitting, the training column has mean near zero and standard deviation near one.

Scaling matters for distance-based models, gradient optimization, and coefficient penalties. It does not make bad data valid.

### Work it out first

Training mean `μ = 50`, standard deviation `σ = 10`.

For `x = 70`:

`z = (70 - 50) / 10`  
`= 20 / 10`  
`= 2`

The value is two standard deviations above the training mean.

### Notebook bridge

The notebook's numerical preprocessing branch fits scaling inside the pipeline.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train[["income"]])

Expected output:

```text
The scaler stores training mean and variance for later transforms.
```


## 4. Encoding Represents Categories Numerically

Models usually require numerical inputs. **One-hot encoding** creates one binary column for each known category.

City values `Colombo`, `Kandy`, and `Galle` become columns such as:

`city_Colombo`, `city_Kandy`, `city_Galle`

One-hot encoding does not impose a false numerical order. The encoder must define what happens when production data contains an unseen category.

### Work it out first

`Kandy` becomes:

`[0, 1, 0]`

The three positions correspond to the fitted category order. If `Jaffna` was unseen, `handle_unknown="ignore"` produces zeros for known category columns.

### Notebook bridge

The categorical branch encodes values using categories learned only from training rows.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore")
encoder.fit(X_train[["city"]])

Expected output:

```text
The encoder stores categories observed in the training city column.
```


## 5. ColumnTransformer Applies Rules by Column Type

A **ColumnTransformer** applies different transformer sequences to selected columns and concatenates their outputs.

Example:

- numerical columns -> median imputer -> standard scaler;
- categorical columns -> most-frequent imputer -> one-hot encoder.

The transformed feature matrix may contain more columns than the raw table because one category column expands into several indicator columns.

### Work it out first

Raw columns:

`age`, `income`, `city`

If city has three fitted categories, transformed columns are approximately:

`scaled_age`, `scaled_income`, `city_A`, `city_B`, `city_C`

Three raw columns become five model inputs.

### Notebook bridge

The notebook uses `ColumnTransformer` to keep feature-specific preprocessing explicit.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer([
    ("numbers", numeric_pipeline, ["age", "income"]),
    ("categories", category_pipeline, ["city"]),
])

Expected output:

```text
One transformer that routes declared columns through two branches.
```


## 6. Pipeline Gives One Fit and Predict Boundary

A scikit-learn **Pipeline** chains transformers and a final estimator.

During `fit(X_train, y_train)`:

1. fit the first transformer;
2. transform training data;
3. fit and transform each next transformer;
4. fit the final estimator.

During `predict(X_new)`:

1. transform with already-fitted transformers;
2. call the already-fitted estimator;
3. return predictions.

### Work it out first

Pipeline:

`preprocess -> LogisticRegression`

One call to `fit()` learns imputers, scaler, encoder, and classifier. One call to `predict()` reuses all learned pieces in order.

### Notebook bridge

The source notebook's saved artifact is a pipeline rather than an estimator detached from preprocessing.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000)),
])

Expected output:

```text
A composite estimator exposing fit, predict, and predict_proba.
```


## 7. Pipelines Prevent Preprocessing Leakage

Unsafe sequence:

1. fit scaler on the complete dataset;
2. transform all rows;
3. split or cross-validate.

Validation information influences the scaler.

Safe sequence:

1. split training and test data;
2. put scaler and estimator inside a pipeline;
3. for each validation fold, fit the complete pipeline only on that fold's training rows;
4. transform the held-out fold with those fitted values.

### Work it out first

Training fold values `[0,10]` have mean `5`. Validation value `[100]` must not change that mean.

If all values are used, mean becomes `(0+10+100)/3 = 36.67`, leaking validation information.

### Notebook bridge

Learners should never preprocess the complete table before notebook cross-validation.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
scores = cross_val_score(
    model, X_train, y_train, cv=5, scoring="roc_auc"
)

Expected output:

```text
Five scores, each produced by fitting the entire pipeline on four folds.
```


## 8. Cross-Validation Measures Variation Across Splits

In **k-fold cross-validation**:

1. split training data into `k` folds;
2. fit on `k-1` folds;
3. validate on the remaining fold;
4. repeat until every fold has served as validation;
5. report scores and their variation.

Cross-validation supports model selection using training data. The untouched test set remains for final evaluation.

Use stratified, grouped, or time-aware splitting when the data structure requires it.

### Work it out first

Five validation scores:

`[0.72, 0.75, 0.68, 0.78, 0.71]`

Mean:

`3.64 / 5 = 0.728`

Range `0.68` to `0.78` shows split-to-split variation.

### Notebook bridge

Learners will compare candidate pipelines using the same split strategy and metric.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
scores = cross_val_score(model, X_train, y_train, cv=5)
print(scores.mean(), scores.std())

Expected output:

```text
The mean validation score and its standard deviation across folds.
```


## 9. Model Comparison Must Use the Same Evidence

A **hyperparameter** is a setting chosen before fitting, such as Ridge `alpha` or tree depth.

For fair comparison:

- use the same training rows and folds;
- fit each candidate's preprocessing inside each fold;
- use the same metric;
- compare baseline and candidate score distributions;
- consider latency, interpretability, memory, and failure behaviour;
- select using validation evidence;
- evaluate the final choice once on test data.

### Work it out first

Cross-validation ROC-AUC:

Baseline `0.50`  
Logistic pipeline `0.74 ± 0.03`  
Tree pipeline `0.75 ± 0.08`

The tree's slightly higher mean comes with greater variation. The correct choice requires more than the largest mean.

### Notebook bridge

The notebook's estimator comparison should preserve identical preprocessing and folds.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    model, {"classifier__C": [0.1, 1, 10]},
    cv=5, scoring="roc_auc"
)

Expected output:

```text
Each C value is evaluated through the complete pipeline across five folds.
```


## 10. Reproducibility Requires More Than a Random Seed

**Reproducibility** means another run can reconstruct the procedure and obtain expected results within known sources of variation.

Record:

- code revision;
- immutable data identity or checksum;
- feature and target schema;
- split method and random seed;
- library and Python versions;
- pipeline parameters;
- metrics for every fold and final test;
- hardware when relevant;
- generated model and reports.

A seed controls some randomness. It does not preserve changing data, code, packages, or hardware behaviour.

### Work it out first

Two runs both use seed `42`, but one uses a changed CSV. Their results differ because the seed does not identify the training data.

### Notebook bridge

Learners should save metadata beside the fitted pipeline and evaluation report.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
experiment = {
    "seed": 42,
    "data_sha256": "...",
    "code_revision": "...",
    "metric": "roc_auc",
}

Expected output:

```text
A structured record connecting results to code, data, settings, and metric.
```


## 11. Persist the Whole Artifact and Load It Carefully

**Serialization** stores a fitted object so it can be loaded later.

Persist:

- complete preprocessing and estimator pipeline;
- input schema;
- label meanings and threshold;
- dependency versions;
- training data reference;
- validation and test evidence;
- artifact checksum.

Pickle-based formats can execute code when loaded. Load only trusted artifacts. Scikit-learn does not support loading a model across arbitrary version changes.

### Work it out first

Saving only `LogisticRegression` loses the fitted encoder and scaler. A production row may then have the wrong number, order, or scale of features.

Saving the complete pipeline preserves the transformation order.

### Notebook bridge

The notebook's save step should persist the full pipeline and its metadata.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
import joblib

joblib.dump(model, "model-pipeline.joblib")
loaded = joblib.load("model-pipeline.joblib")

Expected output:

```text
The loaded trusted artifact exposes the same pipeline prediction interface.
```


## 12. Guided Lab: Build a Leakage-Safe Pipeline

Build and document:

1. numerical and categorical column lists;
2. missing-value policies;
3. numerical scaling;
4. categorical encoding with unknown handling;
5. `ColumnTransformer`;
6. estimator inside `Pipeline`;
7. task-appropriate cross-validation;
8. baseline and two candidate models;
9. selected hyperparameters;
10. untouched test result;
11. serialized complete pipeline and metadata;
12. a prediction after reload.

### Work it out first

If the raw table has `2` numerical columns and one city column with `3` known categories, predict approximately `5` transformed columns. Confirm with `get_feature_names_out()` after fitting.

### Notebook bridge

Complete the pipeline and save sections of `03.model-development.ipynb`.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
model.fit(X_train, y_train)
joblib.dump(model, "pipeline.joblib")
reloaded = joblib.load("pipeline.joblib")
assert np.array_equal(model.predict(X_test), reloaded.predict(X_test))

Expected output:

```text
The assertion passes for a trusted artifact loaded in the same environment.
```


## Guided lab

Build and document:

1. numerical and categorical column lists;
2. missing-value policies;
3. numerical scaling;
4. categorical encoding with unknown handling;
5. `ColumnTransformer`;
6. estimator inside `Pipeline`;
7. task-appropriate cross-validation;
8. baseline and two candidate models;
9. selected hyperparameters;
10. untouched test result;
11. serialized complete pipeline and metadata;
12. a prediction after reload.

### Reference result

If the raw table has `2` numerical columns and one city column with `3` known categories, predict approximately `5` transformed columns. Confirm with `get_feature_names_out()` after fitting.


In [ ]:
# Guided lab workspace: Week 10
# Add only the imports needed for the current step.

# TODO 1: Prepare the smallest valid input.

# TODO 2: Apply the concept taught in this lesson.

# TODO 3: Display inspectable intermediate evidence.

# TODO 4: Compare the result with a hand calculation or stated requirement.

## Weekly deliverable

Submit the completed guided lab with:

- your prediction before execution;
- intermediate values, shapes, metrics, or traces;
- one failed assumption and its correction;
- a plain-English explanation of the result;
- the source notebook section you are now ready to complete.


## Sources and source notebooks

- <https://scikit-learn.org/stable/modules/compose.html>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/03.model-development.ipynb>
- <https://scikit-learn.org/stable/modules/impute.html>
- <https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html>
- <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html>
- <https://scikit-learn.org/stable/modules/preprocessing.html>
- <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html>
- <https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features>
- <https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html>
- <https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html>
- <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html>
- <https://scikit-learn.org/stable/getting_started.html#pipelines-chaining-pre-processors-and-estimators>
- <https://scikit-learn.org/stable/common_pitfalls.html#data-leakage>
- <https://scikit-learn.org/stable/modules/compose.html#pipeline-chaining-estimators>
- <https://scikit-learn.org/stable/modules/cross_validation.html>
- <https://scikit-learn.org/stable/model_selection.html>
- <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html>
- <https://scikit-learn.org/stable/common_pitfalls.html#controlling-randomness>
- <https://mlflow.org/docs/latest/ml/tracking/>
- <https://scikit-learn.org/stable/model_persistence.html>
- <https://joblib.readthedocs.io/en/latest/persistence.html>